# Fourth CuPy example: using ElementwiseKernel

Compute $c_i=a_i+b_i$ with the **same inputs, vector length, and single precision** as the previous notebooks. We now specify a custom operation on one pair of elements through `cp.ElementwiseKernel`. CuPy generates the surrounding indexing and launch logic.

This is the counterpart of the fourth PyCUDA example. The concept is similar, but the interfaces differ: CuPy separates input and output declarations and lets ordinary parameter names denote the current elements, without writing array indices.

Run all cells in order in a Google Colab GPU runtime. Results and optional timings are displayed in cells; no files are exported. An optional final extension computes $2a_i+3b_i$.

## 1. Prepare Google Colab

Select **Runtime → Change runtime type → GPU**, then connect. `nvidia-smi` inspects the remote GPU; its CUDA version describes driver support. `nvcc --version` reports the toolkit compiler and helps select a CuPy package if none is installed.

We do not invoke `nvcc` directly: CuPy generates and compiles the elementwise kernel at runtime. A working CUDA environment is still required.

In [1]:
!nvidia-smi
!nvcc --version

Sat Oct 17 14:09:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Use the existing CuPy installation when available. The next cell installs a CUDA-specific binary package **only if CuPy is absent**, selecting CUDA 12 or 13 from the installed toolkit. It does not install PyCUDA or a notebook extension. Do not install multiple CuPy packages in the same environment.

If an existing CuPy installation fails to import, use a fresh Colab GPU runtime before proceeding. For other environments, follow the [CuPy installation guide](https://docs.cupy.dev/en/stable/install.html).

In [2]:
import importlib.util
import re
import subprocess
import sys

if importlib.util.find_spec("cupy") is None:
    toolkit_info = subprocess.check_output(
        ["nvcc", "--version"], text=True
    )
    match = re.search(r"release\s+(\d+)\.", toolkit_info)
    if match is None or int(match.group(1)) not in (12, 13):
        raise RuntimeError("Use a Colab GPU runtime with CUDA 12 or 13, or follow the CuPy installation guide.")
    package = f"cupy-cuda{match.group(1)}x"
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    print("Installed:", package)
else:
    print("Using the existing CuPy installation.")

Using the existing CuPy installation.


## 2. Initialize CuPy

`cp` is the conventional alias for CuPy. We use device 0 throughout this standalone notebook. Unlike the PyCUDA example, there is no `pycuda.autoinit` import. CUDA failures are reported as Python exceptions.

In [3]:
import numpy as np
import cupy as cp

if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("No CUDA GPU is available. Select a GPU runtime in Colab.")
cp.cuda.Device(0).use()
properties = cp.cuda.runtime.getDeviceProperties(0)
device_name = properties["name"]
if isinstance(device_name, bytes):
    device_name = device_name.decode()
print("GPU:", device_name)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("CuPy:", cp.__version__)
print("CUDA runtime version:", cp.cuda.runtime.runtimeGetVersion())

GPU: Tesla T4
Python: 3.13.15
NumPy: 2.1.3
CuPy: 14.0.1
CUDA runtime version: 13000


## 3. Choose the vector length

Use $N=100003$ as before. `ElementwiseKernel` determines the iteration range from the array shapes and manages the launch. We do not supply a grid or block size. The limit below keeps this notebook's NumPy index construction within its chosen `int32` range.

In [4]:
N = 100003
assert 1 <= N <= np.iinfo(np.int32).max
print(f"Elements: {N}")
print("Launch configuration: managed by ElementwiseKernel.")

Elements: 100003
Launch configuration: managed by ElementwiseKernel.


## 4. Prepare the NumPy inputs

The deterministic inputs match the previous examples: $a_i=0.25(i\bmod101)$ and $b_i=0.50(i\bmod37)$. Both arrays are contiguous `float32` arrays, matching the kernel's `float` pointers.

In [5]:
indices = np.arange(N, dtype=np.int32)
h_a = np.float32(0.25) * (indices % 101).astype(np.float32)
h_b = np.float32(0.50) * (indices % 37).astype(np.float32)
assert h_a.dtype == h_b.dtype == np.float32
assert h_a.flags.c_contiguous and h_b.flags.c_contiguous
print(f"Bytes per vector: {h_a.nbytes}")

Bytes per vector: 400012


## 5. Create the device arrays

Copy the two NumPy inputs with `cp.asarray` and preallocate the output using `cp.empty_like`. All three arrays stay on the GPU. The output is uninitialized because every element will be written by the kernel.

We explicitly pass the output to `ElementwiseKernel`, allowing repeated calls to reuse the same storage. Unlike the third example's `d_c = d_a + d_b`, the call below need not create a new output array each time.

In [6]:
d_a = cp.asarray(h_a)
d_b = cp.asarray(h_b)
d_c = cp.empty_like(d_a)
assert d_a.shape == d_b.shape == d_c.shape == (N,)
assert d_a.dtype == d_b.dtype == d_c.dtype == np.float32
print("Device array class:", type(d_a).__name__)
print("Shape:", d_a.shape, "dtype:", d_a.dtype)
print("Bytes per device array:", d_a.nbytes)

Device array class: ndarray
Shape: (100003,) dtype: float32
Bytes per device array: 400012


## 6. Define and call the elementwise operation

The constructor takes four parts: input declarations, output declarations, a scalar operation written in CUDA C/C++ syntax, and the generated kernel's name.

Here `a`, `b`, and `c` denote the values associated with the current element, so the operation is `c = a + b`. In the PyCUDA example we wrote `c[i] = a[i] + b[i]` with pointer declarations. CuPy generates the indexing for these ordinary elementwise parameters. The kernel's code is generated/compiled as needed on its first invocation.

The call supplies inputs followed by the preallocated output. There is no separate length argument or explicit bounds check. Shapes match in this example, so no broadcasting explanation is needed to follow the sum. Retrieval in the next cell waits for the required GPU work.

In [7]:
elementwise_add = cp.ElementwiseKernel(
    "float32 a, float32 b",
    "float32 c",
    "c = a + b;",
    "vector_add_elementwise",
)

elementwise_add(d_a, d_b, d_c)

array([ 0.  ,  0.75,  1.5 , ..., 15.5 , 16.25, 17.  ], dtype=float32)

## 7. Retrieve and validate every output

`cp.asnumpy` explicitly retrieves the GPU result as a NumPy array. Its default blocking behavior makes that result available to the CPU before returning. We use the same independent NumPy reference and tolerances as the preceding examples and check every element, including the last entries.

In [8]:
h_c = cp.asnumpy(d_c)
h_reference = h_a + h_b
if not np.all(np.isfinite(h_c)):
    raise AssertionError("The GPU result contains nonfinite values.")
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
max_abs_error = float(np.max(np.abs(h_c - h_reference)))
print(f"Validation passed for all {N} elements.")
print(f"Maximum absolute error: {max_abs_error:.3e}")

Validation passed for all 100003 elements.
Maximum absolute error: 0.000e+00


## 8. Display the computed sum

The `c[i] (GPU)` column contains the result actually retrieved from the device. The first 12 and last 3 entries are displayed alongside the NumPy reference, with the same layout as the previous examples.

In [9]:
shown_indices = sorted(set(range(min(12, N))) | set(range(max(0, N - 3), N)))
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'c[i] (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_c[i]:15.6f} {h_reference[i]:12.6f}')

       i         a[i]         b[i]      c[i] (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        0.750000     0.750000
       2     0.500000     1.000000        1.500000     1.500000
       3     0.750000     1.500000        2.250000     2.250000
       4     1.000000     2.000000        3.000000     3.000000
       5     1.250000     2.500000        3.750000     3.750000
       6     1.500000     3.000000        4.500000     4.500000
       7     1.750000     3.500000        5.250000     5.250000
       8     2.000000     4.000000        6.000000     6.000000
       9     2.250000     4.500000        6.750000     6.750000
      10     2.500000     5.000000        7.500000     7.500000
      11     2.750000     5.500000        8.250000     8.250000
  100000     2.500000    13.000000       15.500000    15.500000
  100001     2.750000    13.500000       16.250000    16.250000
  100002     3.000000    14.000000      

## 9. Optional: measure the device interval

After three warm-up calls and synchronization, CUDA events bracket 100 calls writing into the same preallocated output. Compilation, allocation, transfers, validation, and printing are excluded. Synchronizing the stop event makes the measured interval available.

This follows the fourth PyCUDA notebook's timing scope and the first two CuPy notebooks. Short kernels may include GPU idle gaps between Python submissions, and caches may serve repeatedly used data. The third CuPy notebook instead measures host elapsed time including output creation; its numbers cannot directly rank these different scopes.

In [10]:
REPEATS = 100
assert REPEATS > 0
stream = cp.cuda.get_current_stream()
for _ in range(3):
    elementwise_add(d_a, d_b, d_c)
stream.synchronize()

start = cp.cuda.Event()
stop = cp.cuda.Event()
start.record(stream)
for _ in range(REPEATS):
    elementwise_add(d_a, d_b, d_c)
stop.record(stream)
stop.synchronize()
elapsed_ms = cp.cuda.get_elapsed_time(start, stop)
mean_ms = elapsed_ms / REPEATS

# Validate outside the event interval.
h_c = cp.asnumpy(d_c)
if not np.all(np.isfinite(h_c)):
    raise AssertionError("The GPU result contains nonfinite values.")
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
print(f"Total device interval for {REPEATS} launches: {elapsed_ms:.6f} ms")
print(f"Mean device interval per launch: {mean_ms:.6f} ms")
print("Allocation, compilation, transfers and validation are excluded.")
print("Final result validated.")

Total device interval for 100 launches: 3.457024 ms
Mean device interval per launch: 0.034570 ms
Allocation, compilation, transfers and validation are excluded.
Final result validated.


## 10. Optional extension: a linear combination

As in the fourth PyCUDA example, compute $c_i=\alpha a_i+\beta b_i$ with $\alpha=2$ and $\beta=3$. The input declarations now include two scalar coefficients, passed as `np.float32`. **All inputs precede the output** in the CuPy call.

Both multiplications and the addition are expressed inside one generated kernel. No separate full-size arrays are required for the two products. This demonstrates customization, not a measured performance advantage. Floating-point tolerances remain appropriate because the GPU compiler may fuse arithmetic operations.

This cell overwrites `d_c`, validates the new result against the corresponding NumPy expression, and displays selected entries. Rerun the sum and retrieval cells to restore the ordinary addition.

In [11]:
linear_combination = cp.ElementwiseKernel(
    "float32 a, float32 b, float32 alpha, float32 beta",
    "float32 c",
    "c = alpha * a + beta * b;",
    "linear_combination_elementwise",
)
alpha = np.float32(2.0)
beta = np.float32(3.0)
linear_combination(d_a, d_b, alpha, beta, d_c)
h_linear = cp.asnumpy(d_c)
h_linear_reference = alpha * h_a + beta * h_b
if not np.all(np.isfinite(h_linear)):
    raise AssertionError("The GPU result contains nonfinite values.")
np.testing.assert_allclose(h_linear, h_linear_reference, rtol=1e-6, atol=1e-6)
print(f"Linear combination validated for all {N} elements: {alpha:g}*a + {beta:g}*b.")
print(f"Maximum absolute error: {np.max(np.abs(h_linear - h_linear_reference)):.3e}")
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'2a+3b (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_linear[i]:15.6f} {h_linear_reference[i]:12.6f}')

Linear combination validated for all 100003 elements: 2*a + 3*b.
Maximum absolute error: 0.000e+00
       i         a[i]         b[i]     2a+3b (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        2.000000     2.000000
       2     0.500000     1.000000        4.000000     4.000000
       3     0.750000     1.500000        6.000000     6.000000
       4     1.000000     2.000000        8.000000     8.000000
       5     1.250000     2.500000       10.000000    10.000000
       6     1.500000     3.000000       12.000000    12.000000
       7     1.750000     3.500000       14.000000    14.000000
       8     2.000000     4.000000       16.000000    16.000000
       9     2.250000     4.500000       18.000000    18.000000
      10     2.500000     5.000000       20.000000    20.000000
      11     2.750000     5.500000       22.000000    22.000000
  100000     2.500000    13.000000       44.000000    44.000000
  100

## 11. Release the array references

Run this cell when finished. Once the GPU work is complete, remove the array references. CuPy's memory pool may retain released storage for reuse. Rerun from the input-transfer cell to compute again.

In [12]:
cp.cuda.get_current_stream().synchronize()
del d_a, d_b, d_c
print("Device array references released.")

Device array references released.


## What changed from the third example?

Instead of selecting the existing array-addition operation, we supply its scalar formula through `ElementwiseKernel`. CuPy still manages indexing and launching. The simple sum introduces the interface; the optional weighted sum shows how to customize it.

The counterpart in PyCUDA also generates an elementwise kernel, but uses pointer arguments and indexed expressions. In the CuPy form used here, input/output declarations are separate and parameter names denote current element values. Output arguments follow all input arguments when calling the kernel.

Both versions keep arrays on the GPU, retrieve results explicitly, and validate against NumPy. Combining scalar operations within one kernel can avoid separate full-size intermediate arrays, but is not by itself evidence of a speedup.

**Preparation status:** delivered with empty outputs. Python syntax, consistency with preceding examples, and CPU-side input and reference calculations were checked. CUDA compilation and GPU execution were not available in the preparation environment. Run all cells on Colab to obtain actual GPU results and timings.

Reference: [CuPy user-defined kernels](https://docs.cupy.dev/en/stable/user_guide/kernel.html).